In [3]:
import pandas as pd
import json
import base64
from pathlib import Path
from pdf2image import convert_from_path
import mimetypes

from anthropic import Anthropic

# ==========================
# CONFIGURATION (COMPANY GATEWAY)
# ==========================
base_url = "https://anthropic.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjUyNTY1MjksImlhdCI6MTc2NTI1NDcyOSwiYXV0aF90aW1lIjoxNzY1MjU0NzI4LCJqdGkiOiI4ZTU0ZDgwOC05MmU5LTQ3YTYtYWUyZi0zNGQzYjk5MWZmYmIiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6IjRlOTFiMWVhLTY4M2UtNDI1My05Zjg3LTNlYzhlMzAyMmQ1MiIsImF0X2hhc2giOiJ0QzF4NWRuRnhMZWw1X3AzaHFfRlZBIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiI0ZTkxYjFlYS02ODNlLTQyNTMtOWY4Ny0zZWM4ZTMwMjJkNTIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.df9PWvr-r_hg_tZzleq6OZ6a8cSWdCJ403yb2EPQ9dtbNVI8amb5wgkV5Vl29rDNc9VBdZfepi-K4mx5S44SADHVxEcnQnK8gfik-t4FA-ZfjlDxgNdcPQhPSEjCw7tBALCcYPKGcDYWBT8YECuPNExPhuvCX5IY5IQ7zjrGMZWzoq-u2XZcXzhp1HK2KEoEXBuFz3i3tsqN280syj2dpXofiAJHqDLeuKxWp2YMBJN6_mhtEmo5uBeff7OWEKe7F7oFaiVFFNLIl3Ae5SdFIT98ItHdj_MooHB1XK5flk1vXBKtl2EhplM93Ci7WWGZa8QGgUPoAoVMDpRqNM6BkQ"

client = Anthropic(
    base_url=base_url,
    api_key=access_token
)
print("Claude (company gateway) initialized - Extract Only Mode (NO Calculations).")


# ==========================
# PROMPT (SELECTIVE DUAL UNITS)
# ==========================
def get_dimension_prompt():
    return """
    You are an Engineering Dimension Extraction AI specialized in reading dimensioned mechanical
    drawings. You must ONLY extract values that are explicitly written on the drawing.

    ═══════════════════════════════════════════════════════════════════════
    SECTION 1 – TITLE BLOCK METADATA
    ═══════════════════════════════════════════════════════════════════════
    - title: The main drawing title from the title block.
    - drawing_number: The drawing / part / print number from the title block.

    ═══════════════════════════════════════════════════════════════════════
    SECTION 2 – DIMENSION EXTRACTION RULES
    ═══════════════════════════════════════════════════════════════════════
    
    Extract the following parameters from the drawing:
    
    A) SINGLE UNIT PARAMETERS (extract ONE unit only):
       - length
       - inner_diameter (ID)
       - outer_diameter (OD)
       - material (MATL)
    
    B) DUAL UNIT PARAMETERS (extract TWO units if both are shown):
       - surface_area (S/A)
       - weight (WT)
    
    ═══════════════════════════════════════════════════════════════════════
    EXTRACTION FORMAT
    ═══════════════════════════════════════════════════════════════════════
    
    FOR SINGLE UNIT PARAMETERS (length, inner_diameter, outer_diameter, material):
    - Extract ONLY the primary unit shown on the drawing
    - Format: {parameter}_value and {parameter}_unit
    - Even if a second unit is shown in parentheses, IGNORE it
    - Example: If drawing shows "100mm (10cm)", extract only: 100 and "mm"
    
    FOR DUAL UNIT PARAMETERS (surface_area, weight):
    - Extract BOTH units if both are explicitly shown
    - Primary unit: {parameter}_value and {parameter}_unit
    - Alternative unit: {parameter}_value_alt and {parameter}_unit_alt
    - Example: If drawing shows "23.6 in² (152.3 cm²)", extract both
    - If only ONE unit is shown, set _alt fields to null
    
    ═══════════════════════════════════════════════════════════════════════
    CRITICAL RULES (MUST FOLLOW)
    ═══════════════════════════════════════════════════════════════════════
    
    1. READ ONLY EXPLICIT VALUES
       - Extract ONLY values clearly labeled with dimension arrows or annotations
       - DO NOT guess, estimate, measure visually, or calculate
       - DO NOT infer values from the drawing
    
    2. MATERIAL EXTRACTION
       - Look for labels like: MATL, MATERIAL, MAT'L
       - Extract exact text as shown (e.g., "Aluminum 6061", "304 SS", "Steel")
       - If not explicitly labeled, set to null
    
    3. SURFACE AREA EXTRACTION
       - Look for labels like: S/A, Surface Area, SURF AREA
       - Common units: in², cm², m², sq in, square inches
       - Extract BOTH units if shown (e.g., "23.6 in² (152.3 cm²)")
       - If only one unit shown, set _alt to null
    
    4. WEIGHT EXTRACTION
       - Look for labels like: WT, Weight, WEIGHT, Wt.
       - Common units: lb, lbs, kg, pounds, kilograms
       - Extract BOTH units if shown (e.g., "0.52 lb (0.24 kg)")
       - If only one unit shown, set _alt to null
    
    5. NULL HANDLING
       - If a value is not explicitly labeled → set value=null and unit=null
       - If only primary unit exists (for dual parameters) → set _alt=null and _alt_unit=null
       - Missing is better than wrong
    
    ═══════════════════════════════════════════════════════════════════════
    OUTPUT FORMAT (STRICT JSON, NO MARKDOWN)
    ═══════════════════════════════════════════════════════════════════════
    
    Return ONLY this JSON structure:
    
    {
      "title": "SPACER RING",
      "drawing_number": "DRW-12345-A",
      
      "length_value": 100.0,
      "length_unit": "mm",
      
      "inner_diameter_value": 20.0,
      "inner_diameter_unit": "mm",
      
      "outer_diameter_value": 30.0,
      "outer_diameter_unit": "mm",
      
      "material": "Aluminum 6061",
      
      "surface_area_value": 23.6,
      "surface_area_unit": "in^2",
      "surface_area_value_alt": 152.3,
      "surface_area_unit_alt": "cm^2",
      
      "weight_value": 0.52,
      "weight_unit": "lb",
      "weight_value_alt": 0.24,
      "weight_unit_alt": "kg"
    }
    
    FIELD DEFINITIONS:
    - All *_value fields: numeric (float) or null
    - All *_unit fields: string or null
    - material: string or null
    - title: string or null
    - drawing_number: string or null
    
    REMEMBER:
    - For length, inner_diameter, outer_diameter: SINGLE unit only (ignore alt units)
    - For surface_area, weight: DUAL units if both explicitly shown
    - Only extract what you can clearly see and read
    - When in doubt, use null
    """


def cleanup_extracted_data(data: dict) -> dict:
    """
    Clean up extracted data - remove any alt units that shouldn't be there.
    NO CALCULATIONS - only use what the model extracted from the drawing.
    """
    # Explicitly ensure NO alt units for length and diameters
    # (Remove any that might have been extracted by mistake)
    for param in ["length", "inner_diameter", "outer_diameter"]:
        # These should never have alt units - remove them
        if f"{param}_value_alt" in data:
            del data[f"{param}_value_alt"]
        if f"{param}_unit_alt" in data:
            del data[f"{param}_unit_alt"]
    
    # For surface_area and weight, keep whatever was extracted
    # NO automatic conversion - if alt units weren't in the drawing, they stay null
    
    return data


# ==========================
# LOW-LEVEL CALL (image bytes → JSON)
# ==========================
def _call_model_on_image_bytes(image_bytes: bytes, mime_type: str):
    """Call Claude model with image bytes."""
    # Convert image to base64
    image_data = base64.standard_b64encode(image_bytes).decode("utf-8")
    
    # Create message with image
    response = client.messages.create(
        model="claude-opus-4-20250514",  # Latest Claude Sonnet 4.5 model
        max_tokens=4096,
        temperature=0.1,  # Low temperature for consistency
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": mime_type,
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": get_dimension_prompt()
                    }
                ],
            }
        ],
    )

    raw = response.content[0].text

    # Strip code fences if present
    if "```json" in raw:
        raw = raw.split("```json")[1].split("```")[0]
    elif "```" in raw:
        raw = raw.split("```")[1].split("```")[0]

    return json.loads(raw.strip())


# ==========================
# ENHANCED IMAGE ANALYSIS
# ==========================
def analyze_image(image_path: str):
    """Analyze image and extract dimensions with selective dual units."""
    print(f"   → Analyzing: {Path(image_path).name}")

    # Read file as bytes
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    try:
        data = _call_model_on_image_bytes(image_bytes, mime_type)
        print(f"      ✓ Extraction successful")
    except Exception as e:
        print(f"      ✗ Parsing error: {e}")
        data = {}

    # Enforce schema with selective alt fields
    # SINGLE UNIT: length, inner_diameter, outer_diameter, material
    # DUAL UNIT: surface_area, weight
    
    single_unit_params = ["length", "inner_diameter", "outer_diameter"]
    dual_unit_params = ["surface_area", "weight"]
    
    # Initialize all fields
    data.setdefault("title", None)
    data.setdefault("drawing_number", None)
    data.setdefault("material", None)
    
    # Single unit parameters (no _alt fields needed in output)
    for param in single_unit_params:
        data.setdefault(f"{param}_value", None)
        data.setdefault(f"{param}_unit", None)
    
    # Dual unit parameters (with _alt fields)
    for param in dual_unit_params:
        data.setdefault(f"{param}_value", None)
        data.setdefault(f"{param}_unit", None)
        data.setdefault(f"{param}_value_alt", None)
        data.setdefault(f"{param}_unit_alt", None)

    # Clean up data - remove unwanted alt units, NO calculations
    data = cleanup_extracted_data(data)

    return data


# ==========================
# MAIN PROCESSOR
# ==========================
def process_file_enhanced(file_path: str):
    """Process file with enhanced reporting."""
    print(f"\n{'='*70}")
    print(f"DIMENSION EXTRACTOR - SELECTIVE DUAL UNITS MODE")
    print(f"{'='*70}")
    print(f"📄 File: {file_path}")
    
    if not Path(file_path).exists():
        print(f"✗ ERROR: File not found")
        return

    ext = Path(file_path).suffix.lower()
    page_images = []

    # Convert PDF to images
    if ext == ".pdf":
        print(f"\n🔄 Converting PDF to images (300 DPI)...")
        try:
            pages = convert_from_path(file_path, dpi=300)
            if not pages:
                print("✗ ERROR: PDF conversion resulted in 0 pages")
                return
            
            print(f"   ✓ Successfully converted {len(pages)} pages")
            
            for i, p in enumerate(pages, start=1):
                img = f"temp_dim_{i}.png"
                p.save(img, "PNG")
                page_images.append(img)
                
        except Exception as e:
            print(f"✗ ERROR: PDF conversion failed: {e}")
            return
    else:
        page_images = [file_path]
        print(f"   ✓ Processing single image file")

    # Process each page
    print(f"\n{'─'*70}")
    print(f"EXTRACTING DIMENSIONS")
    print(f"{'─'*70}")

    rows = []
    for i, img in enumerate(page_images, start=1):
        print(f"\n📊 Page {i}/{len(page_images)}")
        result = analyze_image(img)
        result["Page"] = i
        
        # Display extracted values
        print(f"      Title: {result.get('title', 'N/A')}")
        print(f"      Drawing #: {result.get('drawing_number', 'N/A')}")
        print(f"      Length: {result.get('length_value')} {result.get('length_unit', '')}")
        print(f"      Inner Diameter: {result.get('inner_diameter_value')} {result.get('inner_diameter_unit', '')}")
        print(f"      Outer Diameter: {result.get('outer_diameter_value')} {result.get('outer_diameter_unit', '')}")
        print(f"      Material: {result.get('material', 'N/A')}")
        
        # Show dual units for surface area
        sa_val = result.get('surface_area_value')
        sa_unit = result.get('surface_area_unit', '')
        sa_val_alt = result.get('surface_area_value_alt')
        sa_unit_alt = result.get('surface_area_unit_alt', '')
        if sa_val:
            sa_str = f"{sa_val} {sa_unit}"
            if sa_val_alt:
                sa_str += f" ({sa_val_alt} {sa_unit_alt})"
            print(f"      Surface Area: {sa_str}")
        
        # Show dual units for weight
        wt_val = result.get('weight_value')
        wt_unit = result.get('weight_unit', '')
        wt_val_alt = result.get('weight_value_alt')
        wt_unit_alt = result.get('weight_unit_alt', '')
        if wt_val:
            wt_str = f"{wt_val} {wt_unit}"
            if wt_val_alt:
                wt_str += f" ({wt_val_alt} {wt_unit_alt})"
            print(f"      Weight: {wt_str}")
        
        rows.append(result)

    # Create DataFrame with proper column order
    column_order = [
        "Page",
        "title",
        "drawing_number",
        "length_value",
        "length_unit",
        "inner_diameter_value",
        "inner_diameter_unit",
        "outer_diameter_value",
        "outer_diameter_unit",
        "material",
        "surface_area_value",
        "surface_area_unit",
        "surface_area_value_alt",
        "surface_area_unit_alt",
        "weight_value",
        "weight_unit",
        "weight_value_alt",
        "weight_unit_alt",
    ]
    
    df = pd.DataFrame(rows)
    df = df[column_order]
    
    # Generate output filename
    out_name = f"{Path(file_path).stem}_Dimensions_Extracted.xlsx"
    df.to_excel(out_name, index=False)

    print(f"\n{'='*70}")
    print(f"✅ SUCCESS! Excel file created:")
    print(f"   📁 {out_name}")
    print(f"   📊 {len(rows)} page(s) extracted")
    print(f"\n💡 Note:")
    print(f"   • Only Surface Area and Weight can have dual units")
    print(f"   • Length and Diameters show single unit only")
    print(f"   • NO automatic calculations - only extracted what's in the drawing")
    print(f"   • If dual units not shown in drawing, alt fields will be empty")

    # Cleanup temp files
    if ext == ".pdf":
        for img in page_images:
            Path(img).unlink(missing_ok=True)
        print(f"\n🧹 Cleaned up temporary files")


# ==========================
# ENTRY POINT
# ==========================
if __name__ == "__main__":
    print("\n" + "="*70)
    print("DIMENSION EXTRACTOR - EXTRACT ONLY MODE")
    print("="*70)
    print("\n🤖 Using Claude Sonnet 4.5 (claude-sonnet-4-20250514)")
    print("\n🔍 Extraction Strategy:")
    print("  • Length: SINGLE unit only (extract what's shown)")
    print("  • Inner Diameter: SINGLE unit only (extract what's shown)")
    print("  • Outer Diameter: SINGLE unit only (extract what's shown)")
    print("  • Material: Text value (extract what's shown)")
    print("  • Surface Area: DUAL units IF BOTH are shown in drawing")
    print("  • Weight: DUAL units IF BOTH are shown in drawing")
    
    print("\n⚠️  IMPORTANT:")
    print("  • NO automatic calculations or conversions")
    print("  • ONLY extracts what is explicitly written on the drawing")
    print("  • If drawing shows '23.6 in²' only → alt unit stays empty")
    print("  • If drawing shows '23.6 in² (152.3 cm²)' → both units extracted")
    
    print("\n" + "─"*70)
    print("Ready to process files.")
    print("\nUsage:")
    print("  process_file_enhanced('your_drawing.pdf')")
    print("\nExample:")
    print("  process_file_enhanced('spacer_ring.pdf')")

Claude (company gateway) initialized - Extract Only Mode (NO Calculations).

DIMENSION EXTRACTOR - EXTRACT ONLY MODE

🤖 Using Claude Sonnet 4.5 (claude-sonnet-4-20250514)

🔍 Extraction Strategy:
  • Length: SINGLE unit only (extract what's shown)
  • Inner Diameter: SINGLE unit only (extract what's shown)
  • Outer Diameter: SINGLE unit only (extract what's shown)
  • Material: Text value (extract what's shown)
  • Surface Area: DUAL units IF BOTH are shown in drawing
  • Weight: DUAL units IF BOTH are shown in drawing

⚠️  IMPORTANT:
  • NO automatic calculations or conversions
  • ONLY extracts what is explicitly written on the drawing
  • If drawing shows '23.6 in²' only → alt unit stays empty
  • If drawing shows '23.6 in² (152.3 cm²)' → both units extracted

──────────────────────────────────────────────────────────────────────
Ready to process files.

Usage:
  process_file_enhanced('your_drawing.pdf')

Example:
  process_file_enhanced('spacer_ring.pdf')


In [4]:
file_path = "Master.png"
#file_path = "https://www.shutterstock.com/image-vector/sketch-bushing-vector-eps10-260nw-111717389.jpg"
process_file_enhanced(file_path)


DIMENSION EXTRACTOR - SELECTIVE DUAL UNITS MODE
📄 File: Master.png
   ✓ Processing single image file

──────────────────────────────────────────────────────────────────────
EXTRACTING DIMENSIONS
──────────────────────────────────────────────────────────────────────

📊 Page 1/1
   → Analyzing: Master.png
      ✓ Extraction successful
      Title: BUSHING BOOM PIVOT
      Drawing #: 6811285
      Length: None None
      Inner Diameter: 1.8504 in
      Outer Diameter: 2.5 in
      Material: 1
      Surface Area: 30.26 in^2 (195.24 cm^2)
      Weight: 1.66 lb (0.76 kg)

✅ SUCCESS! Excel file created:
   📁 Master_Dimensions_Extracted.xlsx
   📊 1 page(s) extracted

💡 Note:
   • Only Surface Area and Weight can have dual units
   • Length and Diameters show single unit only
   • NO automatic calculations - only extracted what's in the drawing
   • If dual units not shown in drawing, alt fields will be empty
